# Legacy 모델 학습 및 평가

원본과 동일하게 전체 데이터로 KMeans, GMM, MeanShift, Agglomerative, Random Forest, Logistic Regression을 학습·평가하고 저장합니다.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.cluster import KMeans
# from sklearn.cluster import DBSCAN
from sklearn.cluster import MeanShift
from sklearn.mixture import GaussianMixture
from sklearn.cluster import AgglomerativeClustering
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from scipy.spatial.distance import euclidean
from threading import Thread


import math
import tkinter as tk
from tkinter import ttk
from PIL import Image, ImageTk  # PIL 라이브러리에서 이미지 로드
from tkinter import PhotoImage
from PIL import Image, ImageTk
from tkinter import messagebox

import random
import time
import re # 정규 표현식
import requests # HTTP 요청 및 응답을 받아오는 모듈
import whois # 도메인의 등록 기간을 파악하는 모듈
import sys # 파이썬 인터프리터의 상태를 확인하는 모듈
import socket
import time
import tld
from urllib.parse import urlparse # URL을 파싱하는 모듈
from urllib.request import urlopen, Request # URL을 열고 읽는 모듈
from bs4 import BeautifulSoup, SoupStrainer # HTML 문서를 파싱하는 모듈
from tld import get_tld # URL에서 도메인을 추출하는 모듈
from tld.exceptions import TldBadUrl, TldDomainNotFound # 도메인 추출 시 예외 처리 모듈
from datetime import datetime, timedelta # 날짜와 시간을 다루는 모듈

In [ ]:
from pathlib import Path


def find_project_root(start):
    """현재 실행 위치에서 프로젝트 루트를 찾습니다."""
    current = Path(start).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "legacy" / "notebooks" / "최종.ipynb").exists():
            return candidate
    raise FileNotFoundError("프로젝트 루트를 찾을 수 없습니다.")


PROJECT_ROOT = find_project_root(Path.cwd())
DATA_PATH = PROJECT_ROOT / "data" / "legacy" / "전체15.csv"
MODEL_DIR = PROJECT_ROOT / "legacy" / "models"
IMAGE_DIR = PROJECT_ROOT / "legacy" / "images"

MODEL_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# 데이터 로드
data = pd.read_csv(DATA_PATH)

X = data.drop(['url', 'label'], axis=1)
y = data['label']

### ▶ KMeans

In [ ]:
# KMeans 클러스터링
kmeans = KMeans(n_clusters=2, n_init=10, random_state=42)
result = kmeans.fit_predict(X)

# 클러스터의 분포 확인
print("클러스터 분포:", np.unique(result, return_counts=True))

# 레이블과 클러스터 결과 비교
label_df = pd.DataFrame({'label': y, 'cluster': result})
print("레이블과 클러스터 결과 비교:")
print(label_df.groupby('label')['cluster'].value_counts())

# 클러스터와 레이블 매핑
cluster_label_mapping = label_df.groupby('cluster')['label'].mean().to_dict()
# 클러스터의 평균 레이블을 기반으로 매핑
cluster_to_label = {cluster: 1 if cluster_label_mapping[cluster] > 0.5 else 0 for cluster in cluster_label_mapping}

# 매핑된 라벨을 사용하여 예측
predicted_labels = [cluster_to_label[cluster] for cluster in result]

# 정확도 출력
print(f'정확도: {accuracy_score(y, predicted_labels) * 100:.2f}%')

# 모델 저장
joblib.dump(kmeans, MODEL_DIR / 'kmeans.pkl')

# 열 이름 출력
print(f"특징값:", X.columns)

### ▶ GMM(가우시안분포)

In [ ]:
# GMM 클러스터링
gmm = GaussianMixture(n_components=2, n_init=10, random_state=42)
gmm.fit(X)
result = gmm.predict(X)

# 클러스터의 분포 확인
print("클러스터 분포:", np.unique(result, return_counts=True))

# 레이블과 클러스터 결과 비교
label_df = pd.DataFrame({'label': y, 'cluster': result})
print("레이블과 클러스터 결과 비교:")
print(label_df.groupby('label')['cluster'].value_counts())

# 클러스터와 레이블 매핑
cluster_label_mapping = label_df.groupby('cluster')['label'].mean().to_dict()
# 클러스터의 평균 레이블을 기반으로 매핑
cluster_to_label = {cluster: 1 if cluster_label_mapping[cluster] > 0.5 else 0 for cluster in cluster_label_mapping}

# 매핑된 라벨을 사용하여 예측
predicted_labels = [cluster_to_label[cluster] for cluster in result]

# 정확도 출력
print(f'정확도: {accuracy_score(y, predicted_labels) * 100:.2f}%')

# 모델 저장
joblib.dump(gmm, MODEL_DIR / 'gmm.pkl')

# 열 이름 출력
print(f"특징값:", X.columns)

### ▶ MeanShift(평균이동)

In [ ]:
# MeanShift 생성 및 훈련
ms = MeanShift(bandwidth=2.27)
result = ms.fit_predict(X)

# 클러스터의 분포 확인
print("클러스터 분포:", np.unique(result, return_counts=True))

# 레이블과 클러스터 결과 비교
label_df = pd.DataFrame({'label': y, 'cluster': result})
print(label_df.groupby('label')['cluster'].value_counts())

# 정확도 출력
accuracy = accuracy_score(y, result)
print(f'정확도:', accuracy * 100)

# 모델 저장
joblib.dump(ms, MODEL_DIR / 'meanshift.pkl')

# 열 이름 출력
print(f"특징값:", X.columns)

### ▶ AgglomerativeCluster(병합군집)

In [ ]:
hierarchical_cluster = AgglomerativeClustering(n_clusters=2)
cluster_labels = hierarchical_cluster.fit_predict(X)

print('군집 결과:', np.unique(cluster_labels, return_counts=True))

label_df = pd.DataFrame(y)
label_df['cluster_label'] = cluster_labels
print(label_df.groupby('label')['cluster_label'].value_counts())

# 정확도 확인
accuracy_score(y, cluster_labels)

# 모델 저장
joblib.dump(hierarchical_cluster, MODEL_DIR / 'agglomerative.pkl')
# 변수 저장
joblib.dump(cluster_labels, MODEL_DIR / 'cluster_labels.pkl')

### ▶ RandomForest

In [ ]:
# 랜덤 포레스트 모델 생성 및 학습
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X, y)

# 전체 데이터에 대한 예측
y_pred = rf_model.predict(X)

# 정확도 확인
accuracy = accuracy_score(y, y_pred)
accuracy_100 = accuracy * 100
print('정확도:', accuracy_100)

# 모델 저장
joblib.dump(rf_model, MODEL_DIR / 'rf.pkl')

# 열 이름 출력
print(f"특징값:", X.columns)

### ▶ LogisticRegression

In [ ]:
# 로지스틱 회귀 모델 생성 및 학습
log_reg = LogisticRegression(random_state=42, max_iter=1000)
log_reg.fit(X, y)

# 전체 데이터에 대한 예측
y_pred = log_reg.predict(X)

# 정확도 확인
accuracy = accuracy_score(y, y_pred)
accuracy_100 = accuracy * 100
print('정확도:', accuracy_100)

# 모델 저장
joblib.dump(log_reg, MODEL_DIR / 'lr.pkl')

# 열 이름 출력
print(X.columns)

## 저장된 모델 정확도 계산 함수


In [ ]:
def calculate_kmeans_accuracy(X, y):
    kmeans = joblib.load(MODEL_DIR / 'kmeans.pkl')
    y_pred_clusters = kmeans.predict(X)
    cluster_label_counts = pd.DataFrame({'cluster': y_pred_clusters, 'label': y})
    cluster_label_mapping = cluster_label_counts.groupby('cluster')['label'].agg(lambda x: x.value_counts().idxmax())
    y_pred_labels = np.array([cluster_label_mapping[cluster] for cluster in y_pred_clusters])
    accuracy = accuracy_score(y, y_pred_labels)
    return accuracy * 100



def calculate_gmm_accuracy(X, y):
    gmm = joblib.load(MODEL_DIR / 'gmm.pkl')
    y_pred_clusters = gmm.predict(X)
    cluster_label_counts = pd.DataFrame({'cluster': y_pred_clusters, 'label': y})
    cluster_label_mapping = cluster_label_counts.groupby('cluster')['label'].agg(lambda x: x.value_counts().idxmax())
    y_pred_labels = np.array([cluster_label_mapping[cluster] for cluster in y_pred_clusters])
    accuracy = accuracy_score(y, y_pred_labels)
    return accuracy * 100



def calculate_meanshift_accuracy(X, y):
    ms = joblib.load(MODEL_DIR / 'meanshift.pkl')
    y_pred = ms.predict(X)
    accuracy = accuracy_score(y, y_pred)
    return accuracy * 100



def calculate_agglomerative_accuracy(X, y):
    agglomerative = joblib.load(MODEL_DIR / 'agglomerative.pkl')
    y_pred = agglomerative.fit_predict(X)
    accuracy = accuracy_score(y, y_pred)
    return accuracy * 100



def calculate_rf_accuracy(X, y):
    rf_model = joblib.load(MODEL_DIR / 'rf.pkl')
    y_pred = rf_model.predict(X)
    accuracy = accuracy_score(y, y_pred)
    return accuracy * 100



def calculate_log_reg_accuracy(X, y):
    log_reg = joblib.load(MODEL_DIR / 'lr.pkl')
    y_pred = log_reg.predict(X)
    accuracy = accuracy_score(y, y_pred)
    return accuracy * 100